# KO CBF Generation
Generate a number of keep-out (KO) regions centered on and around the path with feasible velocities and accelerations which will be used to train the controller while still producing a guaranteed safe output at all times.

In [2]:
import pathlib
import h5py

import numpy as np

import itertools

In [3]:
episodes_train_dir = pathlib.Path("../data/episodes/train")
episodes_test_dir = pathlib.Path("../data/episodes/test")

In [4]:
# load all the episodes from file into memory


def load_from_h5(h5_file_dir):
    episodes = []
    files = []
    for file in h5_file_dir.iterdir():
        if file.name == ".gitkeep":
            continue  # don't try to load this one
        episode = dict()
        with h5py.File(file, "r") as f:
            f.visititems(lambda name, obj: episode.update({name: np.asarray(obj)}))
            episode.update(f.attrs)
        episodes.append(episode)
        files.append(file.name)
    return episodes, files


train_episodes, train_files = load_from_h5(episodes_train_dir)
test_episodes, test_files = load_from_h5(episodes_test_dir)

train_episodes[0]

{'ddx_traj': array([-4.53617623e-01, -4.46520250e-01, -4.39422877e-01, -4.32325504e-01,
        -4.25228131e-01, -4.18130759e-01, -4.11033386e-01, -4.03936013e-01,
        -3.96838640e-01, -3.89741267e-01, -3.82643894e-01, -3.75546522e-01,
        -3.68449149e-01, -3.61351776e-01, -3.54254403e-01, -3.47157030e-01,
        -3.40059657e-01, -3.32962285e-01, -3.25864912e-01, -3.18767539e-01,
        -3.11670166e-01, -3.04572793e-01, -2.97475421e-01, -2.90378048e-01,
        -2.83280675e-01, -2.76183302e-01, -2.69085929e-01, -2.61988556e-01,
        -2.54891184e-01, -2.47793811e-01, -2.40696438e-01, -2.33599065e-01,
        -2.26501692e-01, -2.19404319e-01, -2.12306947e-01, -2.05209574e-01,
        -1.98112201e-01, -1.91014828e-01, -1.83917455e-01, -1.76820082e-01,
        -1.69722710e-01, -1.62625337e-01, -1.55527964e-01, -1.48430591e-01,
        -1.41333218e-01, -1.34235846e-01, -1.27138473e-01, -1.20041100e-01,
        -1.12943727e-01, -1.05846354e-01, -9.87489814e-02, -9.16516086e-02,


In [5]:
num_ko_regions = 1  # number of KO regions to generate for each trajectory

# standard deviations (zero mean unless tuple)
dist_ko_offsets = np.diag([1.0, 1.0])
dist_vel_x = 0.5
dist_vel_y = dist_vel_x
dist_accel_x = 0.2
dist_accel_y = dist_accel_x

dist_radius = 0.4, 0.1  # mean, deviation
dist_vel_radius = 0.2
dist_accel_radius = 0.05

r = np.random.default_rng(42)  # for reproducibility

In [6]:
keep_out_train_dir = pathlib.Path("../data/keep_out/train")
keep_out_test_dir = pathlib.Path("../data/keep_out/test")


# empty these repositories so we have no issues when writing the episodes that
# were generated in the preceding section
for file in itertools.chain(keep_out_train_dir.iterdir(), keep_out_test_dir.iterdir()):
    if file.name == ".gitkeep":
        continue  # don't delete this file
    file.unlink()

num_train_episodes = len(train_episodes)
num_test_episodes = len(test_episodes)

for i in range(num_train_episodes + num_test_episodes):
    episode = (
        train_episodes[i]
        if i < num_train_episodes
        else test_episodes[i - num_train_episodes]
    )

    # first generated all the offsets and properties of the keep-out regions
    ko_traj_offsets = r.multivariate_normal(
        np.zeros(2), dist_ko_offsets, (num_ko_regions)
    )
    ko_vel_x = r.normal(0.0, dist_vel_x, (num_ko_regions))
    ko_vel_y = r.normal(0.0, dist_vel_y, (num_ko_regions))
    ko_accel_x = r.normal(0.0, dist_accel_x, (num_ko_regions))
    ko_accel_y = r.normal(0.0, dist_accel_y, (num_ko_regions))

    ko_radius = r.normal(dist_radius[0], dist_radius[1], (num_ko_regions))
    ko_vel_radius = r.normal(0.0, dist_vel_radius, (num_ko_regions))
    ko_accel_radius = r.normal(0.0, dist_accel_radius, (num_ko_regions))

    # now just need to find random points to place these keep-out regions along
    # the trajectory of the system (with appropriate offsets)

    # NOTE: this is highly inefficient but we just need to run it "once" during
    # dataset generation so it doesn't matter

    unique_rand_idxs = []
    while len(unique_rand_idxs) < num_ko_regions:
        idx = r.integers(0, len(episode["t_traj"]))
        if idx not in unique_rand_idxs:
            unique_rand_idxs.append(idx)

    ko_x = ko_traj_offsets[:, 0] + episode["x_traj"][unique_rand_idxs]
    ko_y = ko_traj_offsets[:, 1] + episode["y_traj"][unique_rand_idxs]

    # stores the results in the corresponding file

    file = (
        keep_out_train_dir.joinpath(train_files[i])
        if i < num_train_episodes
        else keep_out_test_dir.joinpath(test_files[i - num_train_episodes])
    )
    with h5py.File(file, "w") as f:
        f.create_dataset("ko_x", data=ko_x)
        f.create_dataset("ko_y", data=ko_y)
        f.create_dataset("ko_vel_x", data=ko_vel_x)
        f.create_dataset("ko_vel_y", data=ko_vel_y)
        f.create_dataset("ko_accel_x", data=ko_accel_x)
        f.create_dataset("ko_accel_y", data=ko_accel_y)
        f.create_dataset("ko_radius", data=ko_radius)
        f.create_dataset("ko_vel_radius", data=ko_vel_radius)
        f.create_dataset("ko_accel_radius", data=ko_accel_radius)